# Exercise 7

1. **Execute o código abaixo em um arquivo cpp da seguinte maneira**:
    
    * Na pasta ompenmp/notebooks/codigo_externo, crie um aquivo chamado: 
        * "exercise_7.cpp".

    * Clique duas vezes para abrí-lo no editor do VSCode.

    * Copie e cole o código abaixo dentro do arquivo e salve.

    * No terminal Linux, vá até a pasta do aquivo: 
        * cd ompenmp/notebooks/codigo_externo

    * Compile o aquivo:
        * g++ -fopenmp exercise_7.cpp -o exercise_7.exe

    * Execute o programa:
        ./exercise_7.exe

2. **Execute o código no notebook (single thread) e depois fora do notebook (número de threads == número de núcleos).**
    * Insira o código necessário para iniciar a contagem do tempo antes do paralelismo
    * Insira o código necessário para calcular o tempo corrigo após o término do paralelismo
    * Faça 3 execuções do código fora do notebook e, para cada uma, faça:
        * **Varie o valor de "n" e o número de threads**    
        * Imprima os resultados no console
        * Copie o tempo para a célula do tipo markdown que se encontra abaixo da célula de código
    * Compare e comente a variação nos tempos.

3. **Volte a este notebook**:
    * Na célula do tipo markdown abaixo da célula que contém o código a ser executado.

**Obs: dentro do notebook é executada apenas uma thread. Por isso, execute fora do notebook para ver o paralelismo.** 


**O código abaixo apresenta uma falha de segmentação.**

**Tente determinar o que está causando o erro e corrija.**

**Em seguida, execute conforme as instruções acima.**

In [8]:
#include "notebooks_reserved_code/openmp_config.h"
#include <omp.h>
#include <stdio.h>
#include <stdlib.h>

int main () 
{
	const int N=1048;
	int nthreads, tid, i, j;

	double start_time, end_time;

	double **a = (double **) malloc(N * sizeof(double *));
	for (i = 0; i < N; i++)
		a[i] = (double *) malloc(N * sizeof(double));

	
	start_time = omp_get_wtime();
	/* Fork a team of threads with explicit variable scoping */
	#pragma omp parallel shared(nthreads, a) private(i, j, tid)
	{
		/* Obtain/print thread info */
		tid = omp_get_thread_num();
		if(tid == 0) 
		{
			nthreads = omp_get_num_threads();
			printf("Number of threads = %d\n", nthreads);
		}
		printf("Thread %d starting...\n", tid);

		
		/* Each thread works on its own private copy of the array */
		#pragma omp for
		for (i=0; i<N; i++){
			for (j=0; j<N; j++){
			  a[i][j] = tid + i + j;
			}
		}

		/* For confirmation */
		printf("Thread %d done. Last element= %f\n",tid,a[N-1][N-1]);

	}  /* All threads join master thread and disband */

	for (i = 0; i < N; i++)
		free(a[i]);
	free(a);

	end_time = omp_get_wtime();
	printf("Time taken: %f seconds\n", end_time - start_time);

}
main()

Number of threads = 1
Thread 0 starting...
Thread 0 done. Last element= 2094.000000
Time taken: 0.005074 seconds


0

## Resultados:

Cole os resultados das execuções.

| Valor de `N` | Tempo (single-thread) | Tempo (multi-thread) |
|--------------|-----------------------|----------------------|
| 1048    | 0.005074              | 0.008135             | 
| 2096   | 0.036802              | 0.017504             |
| 4192  | 1.034683              | 0.255548             |

A execução com múltiplas threads foi menos eficiente para N = 1048, pois o overhead superou os ganhos do paralelismo. No entanto, com o aumento do tamanho do problema (N = 2096), o tempo já foi reduzido pela metade, indicando melhora. Para N = 4192, o paralelismo foi muito vantajoso, com tempo reduzido em mais de quatro vezes. Isso mostra que, quanto maior o problema, mais compensador é o uso de múltiplas threads.

#### n = 1048 (Single-thread)
![n=1048 single-thread](images/ex7/single1.png)

---

#### n = 1048 (Multi-thread)
![n=1048 multi-thread](images/ex7/multi1.png)

---

#### n = 2096 (Single-thread)
![n=2096 single-thread](images/ex7/single2.png)

---

#### n = 2096 (Multi-thread)
![n=2096 multi-thread](images/ex7/multi2.png)

---

#### n = 4192 (Single-thread)
![n=4192 single-thread](images/ex7/single3.png)

---

#### n = 4192 (Multi-thread)
![n=4192 multi-thread](images/ex7/multi3.png)


Cole também o código corrigido.

O problema está na declaração da variável `a` como privada para cada thread, porém a variável é uma matriz grande alocada na stack, e isso gera stack overflow que provoca o erro de segmentação. A matriz `a` tem tamanho de 1048 x 1048 de double, o que da aproximadamente 8,7MB de memória, e como cada thread cria sua própria cópia da matriz na stack, ultrapassa o limite padrão de tamanho.


```c
#include "notebooks_reserved_code/openmp_config.h"
#include <omp.h>
#include <stdio.h>
#include <stdlib.h>

int main () 
{
	const int N=4192;
	int nthreads, tid, i, j;

	double start_time, end_time;

	double **a = (double **) malloc(N * sizeof(double *));
	for (i = 0; i < N; i++)
		a[i] = (double *) malloc(N * sizeof(double));

	
	start_time = omp_get_wtime();
	/* Fork a team of threads with explicit variable scoping */
	#pragma omp parallel shared(nthreads, a) private(i, j, tid)
	{
		/* Obtain/print thread info */
		tid = omp_get_thread_num();
		if(tid == 0) 
		{
			nthreads = omp_get_num_threads();
			printf("Number of threads = %d\n", nthreads);
		}
		printf("Thread %d starting...\n", tid);

		
		/* Each thread works on its own private copy of the array */
		#pragma omp for
		for (i=0; i<N; i++){
			for (j=0; j<N; j++){
			  a[i][j] = tid + i + j;
			}
		}

		/* For confirmation */
		printf("Thread %d done. Last element= %f\n",tid,a[N-1][N-1]);

	}  /* All threads join master thread and disband */

	for (i = 0; i < N; i++)
		free(a[i]);
	free(a);

	end_time = omp_get_wtime();
	printf("Time taken: %f seconds\n", end_time - start_time);

}
main()

```

## Entrega:

**Salve este notebook com as suas respostas e poste como entrega da atividade no Canvas.**